In [1]:
import pickle

data_path = "/Users/ford/Documents/coding_trae/cro_rmi_improvement_feature/src/cro_rmi_improvement_feature/network_analyzer/data/graph/graph_data_library_no_embedding_dict.pkl"
data = pickle.load(open(data_path, "rb"))
data.keys()
company_graph_datas = data["company_graph_datas"]
company_graph_datas.keys()

dict_keys(['PCG|embedding_raw_user_data|oneway_run', 'PCG|embedding_risk_desc_catalog|oneway_run', 'PCG|embedding_summary_user_data|oneway_run', 'PCG|embedding_raw_user_data|twoway_run', 'PCG|embedding_risk_desc_catalog|twoway_run', 'PCG|embedding_summary_user_data|twoway_run', 'lotus_south|embedding_raw_user_data|oneway_run', 'lotus_south|embedding_risk_desc_catalog|oneway_run', 'lotus_south|embedding_summary_user_data|oneway_run', 'lotus_south|embedding_raw_user_data|twoway_run', 'lotus_south|embedding_risk_desc_catalog|twoway_run', 'lotus_south|embedding_summary_user_data|twoway_run'])

In [11]:
import pickle
import networkx as nx
import pandas as pd
import os


# from utils import get_number_edges_to_show
def get_number_edges_to_show(total_nodes):
    return 2 * total_nodes


def filter_non_arrow_edges(edges, slider_value):
    new_edges = []
    for edge in edges:
        if edge["interdependency_type"] == "Causal":
            if edge["cosine_similarity"] >= slider_value:
                new_edges.append(edge)
    return new_edges


debug_list = []
all_data_list = []
for company, data in company_graph_datas.items():
    if company != "PCG|embedding_risk_desc_catalog|oneway_run":
        continue
    G = nx.DiGraph()
    print(data.keys())
    nodes = data["nodes"]
    edges = data["edges"]
    print(edges[0].keys())
    line_weights = [edge["cosine_similarity"] for edge in edges]
    num_edges_to_show = get_number_edges_to_show(len(nodes))
    sorted_weights = sorted(line_weights, reverse=True)
    # The threshold is the weight of the (num_edges_to_show)-th edge (0-indexed)
    slider_value = sorted_weights[num_edges_to_show - 1]
    filter_edges = filter_non_arrow_edges(edges, slider_value)
    print(len(filter_edges))
    for edge in filter_edges:
        G.add_edge(
            edge["source"],
            edge["target"],
            weight=edge["cosine_similarity"],
        )
        if (
            edge["source"] == "risk_PCG_20250513_24"
            or edge["target"] == "risk_PCG_20250513_24"
        ):
            debug_list.append(edge)
    in_degree_centrality_dict = nx.in_degree_centrality(G)
    out_degree_centrality_dict = nx.out_degree_centrality(G)
    betweenness_dict_weight = nx.betweenness_centrality(G, weight="weight")
    betweenness_dict_non_weight = nx.betweenness_centrality(G)

    # create list of data so I can convert to dataframe later
    for node in nodes:
        row_data = {
            "company": company,
            "risk_id": node["data"]["id"],
            "risk_name": node["data"]["label"],
            "risk_level": node["data"]["risk_level"],
            "in_degree": G.in_degree(node["data"]["id"]),
            "out_degree": G.out_degree(node["data"]["id"]),
            "in_degree_centrality": in_degree_centrality_dict.get(
                node["data"]["id"], None
            ),
            "out_degree_centrality": out_degree_centrality_dict.get(
                node["data"]["id"], None
            ),
            "betweenness_centrality_weight": betweenness_dict_weight.get(
                node["data"]["id"], None
            ),
            "betweenness_centrality_non_weight": betweenness_dict_non_weight.get(
                node["data"]["id"], None
            ),
        }
        all_data_list.append(row_data)
# num_edges_to_show = get_number_edges_to_show(total_nodes)

# if num_edges_to_show == 0:
#     # If showing 0 edges, set threshold higher than max weight
#     slider_value = max(line_weights) + 1 if line_weights else float("inf")
# elif num_edges_to_show == total_edges:
#     # If showing all edges, set threshold lower than min weight
#     slider_value = min(line_weights) - 1 if line_weights else float("-inf")
# else:
#     # Sort weights descending and find the weight at the index corresponding to the number of edges
#     sorted_weights = sorted(line_weights, reverse=True)
#     # The threshold is the weight of the (num_edges_to_show)-th edge (0-indexed)
#     slider_value = sorted_weights[num_edges_to_show - 1]
all_data_df = pd.DataFrame(all_data_list)
# sort in_degree_centrality descending
all_data_df = all_data_df.sort_values(by="in_degree_centrality", ascending=False)
all_data_df.head()

dict_keys(['nodes', 'edges', 'number_of_displayed_edges', 'risk_catalog_reference_id', 'overlay_top_n_risks_catalog_id', 'overlay_news_id_list'])
dict_keys(['source', 'target', 'interdependency_type', 'direction', 'rationale', 'confidence', 'risk_a_data', 'risk_b_data', 'distance', 'cosine_similarity', 'high_priority', 'similarity_rank'])
54


,company,risk_id,risk_name,risk_level,in_degree,out_degree,in_degree_centrality,out_degree_centrality,betweenness_centrality_weight,betweenness_centrality_non_weight
58,PCG|embedding_risk_desc_catalog|oneway_run,risk_PCG_20250513_58,Unable to deliver product,2,9,1,0.236842,0.026316,0.028450,0.027620
36,PCG|embedding_risk_desc_catalog|oneway_run,risk_PCG_20250513_36,Operational inefficiency,2,5,3,0.131579,0.078947,0.012091,0.012328
41,PCG|embedding_risk_desc_catalog|oneway_run,risk_PCG_20250513_41,Poor service quality,1,4,2,0.105263,0.052632,0.023471,0.023945
40,PCG|embedding_risk_desc_catalog|oneway_run,risk_PCG_20250513_40,Poor product quality,2,4,3,0.105263,0.078947,0.010669,0.010194
54,PCG|embedding_risk_desc_catalog|oneway_run,risk_PCG_20250513_54,Service-related dissatisfaction,2,4,1,0.105263,0.026316,0.002845,0.003793


In [12]:
# filter to have only company "lotus_south|embedding_risk_desc_catalog|oneway_run"
select_company_df = all_data_df[
    all_data_df["company"] == "PCG|embedding_risk_desc_catalog|oneway_run"
]
# select_company_df sort risk_level descending
select_company_df = select_company_df.sort_values(by="risk_level", ascending=False)
select_company_df

,company,risk_id,risk_name,risk_level,in_degree,out_degree,in_degree_centrality,out_degree_centrality,betweenness_centrality_weight,betweenness_centrality_non_weight
24,PCG|embedding_risk_desc_catalog|oneway_run,risk_PCG_20250513_24,Intense market competition,4,1,1,0.026316,0.026316,0.000711,0.000711
58,PCG|embedding_risk_desc_catalog|oneway_run,risk_PCG_20250513_58,Unable to deliver product,2,9,1,0.236842,0.026316,0.028450,0.027620
1,PCG|embedding_risk_desc_catalog|oneway_run,risk_PCG_20250513_1,Business interruption from fire hazards,2,0,1,0.000000,0.026316,0.000000,0.000000
38,PCG|embedding_risk_desc_catalog|oneway_run,risk_PCG_20250513_38,Overstocking inventory,2,0,2,0.000000,0.052632,0.000000,0.000000
37,PCG|embedding_risk_desc_catalog|oneway_run,risk_PCG_20250513_37,Outsourcing inefficiency,2,0,5,0.000000,0.131579,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
60,PCG|embedding_risk_desc_catalog|oneway_run,risk_PCG_20250513_60,Understocking inventory,1,1,2,0.026316,0.052632,0.000000,0.000000
8,PCG|embedding_risk_desc_catalog|oneway_run,risk_PCG_20250513_8,Debtor credit risk (Internal party),1,(),(),NaN,NaN,NaN,NaN
7,PCG|embedding_risk_desc_catalog|oneway_run,risk_PCG_20250513_7,Debtor credit risk (External party),1,(),(),NaN,NaN,NaN,NaN
39,PCG|embedding_risk_desc_catalog|oneway_run,risk_PCG_20250513_39,Poor internal cooperation,1,0,1,0.000000,0.026316,0.000000,0.000000


In [4]:
all_data_df.shape

(66, 10)

In [5]:
len(debug_list)

0

In [6]:
for i in debug_list:
    print(i.keys())
    print(i["risk_a_data"]["risk"])
    print(i["risk_b_data"]["risk"])
    print(i["similarity_rank"])
    print(i["cosine_similarity"])
    print(i["interdependency_type"])
    print(i["direction"])

    print()

In [7]:
slider_value

0.6110811081796713

In [8]:
len(filter_edges)

0